[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/barmag/fanous-llm-lens/blob/main/notebooks/education/stage2_dash_skip_trigram_reference.ipynb)

In [ ]:
# Setup: install deps + fetch the shared helper on Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q transformer_lens tokenizers huggingface_hub plotly
    # transformer_lens pulls a newer torch; Colab's pre-installed torchaudio was built
    # against the old torch and its .so then fails to load (undefined symbol:
    # torch_library_impl). transformers imports torchaudio lazily and crashes on it,
    # though nothing here uses audio. Remove it so that import path is skipped.
    !pip uninstall -y -q torchaudio
    !wget -q https://raw.githubusercontent.com/barmag/fanous-llm-lens/main/notebooks/education/tiny.py
import tiny
import torch
import plotly.graph_objects as go

torch.manual_seed(42)
print("device:", tiny.device())

<div dir="rtl">

# المرحلة ٢داش: طبقة انتباه واحدة = خليط من bigram و skip-trigram (على عربي حقيقي)

في المرحلة ٢أ ضفنا كتلة انتباه واحدة وشفنا **شكل** الانتباه. هنا بنوصل للنتيجة الأساسية في ورقة *A Mathematical Framework for Transformer Circuits* (Elhage et al., 2021):

> **محوّل بطبقة واحدة، انتباه بس (من غير MLP)، بيتفكّك بالظبط لمجموع:**
> - **مسار مباشر (bigram):** بيتنبأ من التوكن الحالي بس — من غير سياق.
> - **دوائر skip-trigram (رأس لكل رأس):** بتبص لتوكن أقدم في السياق وتنسخ منه معلومة.

والخليط البسيط ده **معبّر بشكل مفاجئ**. عشان الدوائر تبان **مقروءة** مش ضوضاء، الموديل هنا **مدرّب على نطاق محترم**: ٨ رؤوس، d_model=512، على ~٣٤٠ مليون توكن عربي (فصحى أساسًا + مصري). درّبناه مرة واحدة بسكربت `train_stage2dash.py`؛ النوتبوك ده بيحمّل النتيجة ويفكّكها.

**الطريقة أمينة:** في §٥ ما نختارش ثلاثيات لطيفة بإيدينا — بنبحث بذور تصنيفات **محدّدة مقدّمًا** + بحث غير موجَّه، وكل ثلاثي بيتأكّد على تمريرة محتجوزة (lift > ٠) قبل ما نعدّه. بنعدّ اللي اتأكّد فعلًا، والتصنيف الفاضي أو الضعيف **نتيجة** برضه.

**حدود التجربة:** الكوربَس فصحى-أغلب (المصري نادر)، وبذرة واحدة. التفكيك نفسه **مظبوط رياضيًا** لأي موديل من النوع ده.

</div>

<div dir="ltr">

# Stage 2dash: one attention layer = a bigram + skip-trigram ensemble (on real Arabic)

The central result of *A Mathematical Framework for Transformer Circuits* (Elhage et al., 2021):

> **A one-layer attention-only transformer (no MLP) decomposes *exactly* into a sum of:**
> - **a direct path (bigram):** predicts from the current token alone — no context;
> - **skip-trigram circuits (one per head):** look back at an earlier token and copy from it.

This simple ensemble is **surprisingly expressive**. So the circuits read as signal rather than noise, the model here is trained at a **respectable scale**: 8 heads, d_model=512, on ~340M Arabic tokens (mostly MSA + some Masri). It was trained once by `train_stage2dash.py`; this notebook **loads** the result and decomposes it.

**The method is honest:** §5 does *not* hand-pick pretty triples — it searches **pre-committed
seed categories** plus an unsupervised scan, and every triple is checked on a held-out forward
pass (lift > 0) before it counts. We report *verified counts*, and an empty-or-weak category is
itself a finding.

**Limits:** the corpus is MSA-heavy (Masri is scarce) and it's a single seed. The decomposition itself is **mathematically exact** for any such model.

</div>

<div dir="rtl">

## ١. التفكيك بالظبط · The exact decomposition

مخرجات الموديل (logits) بتساوي بالظبط:

$$\text{logits} = \underbrace{x\,W_U}_{\text{bigram}} \;+\; \underbrace{\sum_{\text{heads, positions}} a_{\text{src}}\;(x_{\text{src}}\,W_V W_O W_U)}_{\text{skip-trigram}}$$

الموديل **من غير LayerNorm** و**shortformer** (الموضع في مسار QK بس)، فالتفكيك يطلع مظبوط بالظبط والمسار المباشر يفضل bigram حقيقي.

</div>

<div dir="ltr">

## 1. The exact decomposition

The model's logits equal, exactly, a **bigram direct path** `x·W_U` plus a sum of per-head
**skip-trigram** terms `aₛ·(xₛ·W_V·W_O·W_U)`. The model is **LayerNorm-free** with a
**shortformer** positional embedding (position in the QK path only), so the split is exact
and the direct path stays a true bigram.

</div>

<div dir="rtl">

## ٢. نحمّل الموديل والمُجزّئ · Load the model & tokenizer

نحمّل نقطة الحفظ المدرّبة (محليًا، وإلا من Hugging Face Hub) — موديل بطبقة واحدة، انتباه بس، LN-free + shortformer، مع BPE عربي صغير (~١٢ ألف). للاختبار السريع في الـ CI نبني نسخة مصغّرة (FORCE_TINY).

</div>

In [ ]:
import os

CKPT_DIR = os.path.join(os.path.dirname(os.path.abspath(tiny.__file__)), "checkpoints", "stage2dash")
HF_REPO = "yassermakram/fanous-stage2dash-attn-only-1l"   # push once from train_stage2dash.py

# A small inline Arabic paragraph used for all eval/forward passes (no network needed,
# in-vocab for the trained model). Masri + MSA mix.
EVAL_TEXT = (
    "القطة بتاكل السمك والولد بيشرب اللبن في البيت. "
    "الجو حلو النهارده واحنا رايحين نتمشى في الشارع. "
    "المدينة كبيرة وفيها ناس كتير بتروح وتيجي كل يوم. "
    "هو قال إنه هيسافر بكرة بدري عشان الشغل المهم."
)

def _model_from_ckpt(ckpt):
    c = ckpt["config"]
    model = tiny.make_tiny_model(
        n_layers=c["n_layers"], n_heads=c["n_heads"], d_vocab=c["d_vocab"],
        n_ctx=c["n_ctx"], d_model=c["d_model"], attn_only=c["attn_only"],
        normalization_type=c["normalization_type"],
        positional_embedding_type=c["positional_embedding_type"])
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    return model

def _load_real():
    from tokenizers import Tokenizer
    tpath, mpath = os.path.join(CKPT_DIR, "tokenizer.json"), os.path.join(CKPT_DIR, "model.pt")
    if not (os.path.exists(tpath) and os.path.exists(mpath)):
        try:
            from huggingface_hub import hf_hub_download
            tpath = hf_hub_download(HF_REPO, "tokenizer.json")
            mpath = hf_hub_download(HF_REPO, "model.pt")
        except Exception as e:
            raise FileNotFoundError(
                f"No local checkpoint in {CKPT_DIR} and HF repo {HF_REPO!r} is unavailable "
                f"({type(e).__name__}). Either train it once with `python train_stage2dash.py` "
                f"(writes the checkpoint locally), set HF_REPO to a pushed model "
                f"(train_stage2dash.py --push-hub --hf-repo <you>/...), or set FORCE_TINY=True "
                f"for a quick structural demo on a tiny random model.") from e
    ckpt = torch.load(mpath, map_location=tiny.device(), weights_only=False)  # carries a config dict
    return _model_from_ckpt(ckpt), Tokenizer.from_file(tpath)

def _build_tiny():
    # Fast, network-free stand-in for CI: a tiny BPE on EVAL_TEXT + a tiny random model.
    from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
    tok = Tokenizer(models.BPE(unk_token="[UNK]"))
    tok.normalizer = normalizers.NFKC()
    tok.pre_tokenizer = pre_tokenizers.Whitespace()
    tok.train_from_iterator([EVAL_TEXT] * 20,
                            trainers.BpeTrainer(vocab_size=200, special_tokens=["[UNK]"]))
    model = tiny.make_tiny_model(n_layers=1, n_heads=2, d_vocab=tok.get_vocab_size(),
        n_ctx=32, d_model=64, attn_only=True,
        normalization_type=None, positional_embedding_type="shortformer")
    model.eval()
    return model, tok

def load_model_and_tokenizer():
    return _build_tiny() if globals().get("FORCE_TINY") else _load_real()

model, tok = load_model_and_tokenizer()
VOCAB = tok.get_vocab_size()
id_to_str = {i: (tok.id_to_token(i) or "[?]") for i in range(VOCAB)}
def encode(text):
    return tok.encode(text).ids

print(f"layers=1 heads={model.cfg.n_heads} d_model={model.cfg.d_model} "
      f"n_ctx={model.cfg.n_ctx} vocab={VOCAB}")

<div dir="rtl">

## ٢ب. بذور التصنيفات · Category seeds

عشان ما نختارش الثلاثيات بإيدينا (وندّعي اللي عايزينه)، بنحدّد **مقدّمًا** أربع تصنيفات بذرة بنبحث عنها بشكل منهجي في §٥: تعابير فصحى ثابتة، صيغ دينية، أداة التعريف/الصرف، وتباين فصحى↔مصري. **النسخ الذاتي (self-copy) هو الضابط.** كل ثلاثي بيتأكّد على تمريرة محتجوزة (lift > ٠) قبل ما نعرضه — مش بنعدّ رقم ثابت، بنعدّ اللي اتأكّد فعلًا، والتصنيف الفاضي **نتيجة** برضه.

</div>

<div dir="ltr">

## 2b. Category seeds

To avoid hand-picking triples (and claiming whatever we hoped to find), we fix **four seed
categories up front** and search them systematically in §5: MSA fixed expressions, religious
formulae, the definite article / morphology, and an MSA↔Masri contrast. **Self-copy is the
control.** Every triple is checked on a held-out forward pass (lift > 0) before we show it —
we report *verified counts, not a fixed number*, and an **empty category is itself a finding**.

</div>

In [ ]:
# Category seed lexicons (Arabic-reader curated; edit freely). Self-copy is the control.
# Experiment notebooks can override SEED_LEXICONS (and CKPT_DIR above) before the §5 cells.
SEED_LEXICONS = {
    "MSA fixed expressions": ["الرغم", "بالإضافة", "حين", "بسبب", "أجل"],
    "Religious / formulaic": ["الله", "صلى", "رضي", "شاء", "عليه"],
    "Definite-article / morphology": ["ال", "الذي", "التي"],
    "MSA↔Masri contrast": ["اللي", "عايز", "دلوقتي", "الذي", "يريد", "الآن"],
}
print({k: len(v) for k, v in SEED_LEXICONS.items()})

<div dir="rtl">

## ٣. نثبت التفكيك بإيدينا · The decomposition, by hand

نفكّك الـ logits لتتابع حقيقي لجزئين — **المباشر** (تضمين × `W_U`) و**الـ skip-trigram** (مخرج الانتباه × `W_U`) — ومجموعهم لازم يطابق مخرج الموديل بالظبط: **الفرق ~صفر**.

</div>

In [ ]:
ids = torch.tensor([encode(EVAL_TEXT)[: model.cfg.n_ctx]]).to(tiny.device())
logits, cache = model.run_with_cache(ids)

direct = cache["resid_pre", 0] @ model.W_U + model.b_U   # bigram path       <- (1, ctx, V)
skip   = cache["attn_out", 0] @ model.W_U                # skip-trigram path  <- (1, ctx, V)
print("max |(direct + skip) - logits|:", float((direct + skip - logits).abs().max()))  # ~0

# how big is the skip-trigram contribution relative to the bigram skeleton?
print("||skip|| / ||direct|| (logit space):", round(float(skip.norm() / direct.norm()), 2))

<div dir="rtl">

## ٤. دائرة الـ bigram (المسار المباشر) · The bigram circuit

`W_E W_U` هي **هيكل الـ bigram**: لكل توكن، إيه التوكنز اللي بيرجّحها مباشرة من غير سياق. التوكنز الفصحى المتكررة بتطلع **متلازمات نظيفة** (على الرغم، من أجل، البيت الأبيض). التوكنز المصري (زي "اللي") **ضعيفة** لأن الكوربَس فصحى-أغلب — وده بنفسه ملاحظة عن ندرة بيانات اللهجة.

</div>

<div dir="ltr">

## 4. The bigram circuit

`W_E W_U` is the **bigram skeleton**: for each token, what it directly favours next, with no
context. Frequent MSA tokens give **clean collocations**; Masri tokens (e.g. "اللي") are
**weak** because the corpus is MSA-heavy — itself an observation about dialect-data scarcity.

</div>

In [ ]:
bigram = (model.W_E @ model.W_U).detach().cpu()   # (V, V): current token -> next token

def bigram_top(word, k=6):
    seq = encode(word)
    if not seq:
        return f"{word!r}: empty"
    tid = seq[-1]
    _, idx = torch.topk(bigram[tid], k)
    return f"{id_to_str[tid]:>8} ->  " + " · ".join(id_to_str.get(int(j), "[?]") for j in idx)

print("MSA (well-trained) — crisp collocations:")
for w in ["في", "من", "على", "كان", "البيت"]:
    print(" ", bigram_top(w))
print("\nMasri (data-starved here; corpus is MSA-heavy) — weaker / fragmented:")
for w in ["اللي", "عايز", "دلوقتي"]:
    print(" ", bigram_top(w))

<div dir="rtl">

## ٥. دوائر الـ skip-trigram — الطريقة الأمينة · The honest method

كل واحد من الـ ٨ رؤوس = جدول skip-trigram من دائرتين:
- **QK** = `W_E W_Q W_K^T W_E^T`: لتوكن وجهة، **هنبص على مين** (الـ source)؟
- **OV** = `W_E W_V W_O W_U`: الـ source ده، **إيه الناتج اللي بيرجّحه**؟

الطريقة الأمينة من ٣ خطوات: **(١) تشخيص** — أي رأس بيبص بالمحتوى أصلًا (رأس موضعي مايقدرش يستضيف skip-trigram محتوى)؛ **(٢) تجميع مرشّحين** من بذور التصنيفات + بحث غير موجَّه، مرتّبين بـ `OV·QK`؛ **(٣) تأكيد محتجوز**: نبني `[source … source … dest]` ونشوف هل الموديل الكامل بيرفع `P(output)` فوق خط الـ bigram (lift > ٠). بنعرض **اللي اتأكّد بس**.

**حدود مقدّمًا:** التصنيف بيتعمل على **جملة تشخيص واحدة**؛ والـ BPE بيكسّر البذور (عايز→عا، الذي بتتقسم) فبعض التصنيفات بتطلع ضعيفة. وفي **عيب بنيوي** (§٥ج): رأس واحد مايقدرش يشترط على الـ source والـ dest مع بعض.

</div>

<div dir="ltr">

## 5. The skip-trigram circuits — the honest method

Each of the 8 heads is a skip-trigram table from two circuits: **QK** (`W_E W_Q W_Kᵀ W_Eᵀ`,
which source to attend to) and **OV** (`W_E W_V W_O W_U`, what the attended source promotes).

The honest pipeline is three steps: **(1) diagnose** which heads attend by *content* at all (a
positional head — BOS or previous-token — can't host a content skip-trigram); **(2) pool
candidates** from the seed categories *and* an unsupervised scan, ranked by `OV·QK`; **(3)
verify held-out**: build `[source … source … dest]` and check the full model raises
`P(output)` above the bigram baseline (lift > 0). We print **only verified triples**.

**Limits up front:** the head-kind label is from **one diagnosis sentence**; BPE fragments the
seeds (عايز→عا, الذي splits), so some categories come back weak/empty. And a **structural bug**
(§5c) bounds it: a single head cannot jointly condition on *both* source and destination.

</div>

In [ ]:
import skip_trigrams as st

# §5a — diagnose each head on ONE sentence. (bos/prev are the raw masses the classifier
# thresholds at 0.5; printing them keeps borderline heads honest.)
sent = "هو قال إنه هيسافر بكرة بدري عشان الشغل المهم"
sids = encode(sent)[: model.cfg.n_ctx]
_, hcache = model.run_with_cache(torch.tensor([sids]).to(tiny.device()))
print("Per-head attention kind (positional heads can't host content skip-trigrams):")
for h in range(model.cfg.n_heads):
    patt = hcache["pattern", 0][0, h].detach().cpu()
    n = patt.shape[0]
    den = max(n - 1, 1)
    bos = sum(float(patt[i, 0]) for i in range(1, n)) / den
    prev = sum(float(patt[i, i - 1]) for i in range(1, n)) / den
    print(f"  head {h}: {st.head_attention_kind(patt):>10}   (bos={bos:.2f}  prev={prev:.2f})")

In [ ]:
# §5b — pool ~100 candidates PER CATEGORY, verify the WHOLE pool held-out, keep survivors.
# Each seed source expands to its top-5 promoted outputs x top-4 destinations (5 seeds -> ~100),
# pooled across every content head, deduped, then EVERY candidate is verified.
# CAVEAT (the honest catch, see §7/§8): per_source_dests=4 pairs each source->output OV-promotion
# with up to 4 destinations, and most are spurious high-QK *fragments* (حض, فض, وض). Because
# OV[source] is destination-independent (§5c), the SAME promotion verifies under several dests,
# so the verified COUNT inflates the number of distinct findings. We therefore also track distinct
# source->output, and (§7) only feature triples whose destination is a real co-occurring token.
import skip_trigrams as st

FREQ = min(2500, VOCAB)
CONTENT_HEADS = [h for h in range(model.cfg.n_heads)
                 if st.head_attention_kind(
                     hcache["pattern", 0][0, h].detach().cpu()) == "content"] or [0]

def pool_for(seed_words, heads, want=100):
    src = set(st.seed_ids(encode, id_to_str, seed_words, FREQ)) if seed_words else None
    per_out, per_dst = (5, 4) if seed_words else (1, 1)   # seeds expand; unsupervised scans broad
    cands = []
    for h in heads:
        for v in st.candidate_pool(model, head=h, freq=FREQ, top_n=want, sources=src,
                                   per_source_outputs=per_out, per_source_dests=per_dst):
            cands.append({**v, "head": h})
    cands = st.dedup_triples(cands)                       # same triple from two heads -> one
    cands.sort(key=lambda c: c["score"], reverse=True)
    cands = cands[:want]                                  # the ~100 we pick from
    verified = [st.verify_triple(model, c) for c in cands]
    verified.sort(key=lambda v: v["lift"], reverse=True)
    return verified

def show(label, verified, k=6):
    real = [v for v in verified if v["verified"]]
    strong = [v for v in real if v["lift"] >= 0.05]
    print(f"\n{label}: {len(real)} verified ({len(strong)} with lift>=0.05) of {len(verified)} checked")
    for v in real[:k]:
        s, d, o = id_to_str.get(v['source']), id_to_str.get(v['dest']), id_to_str.get(v['output'])
        print(f"   h{v['head']} [{s} ... {d} -> {o}]   lift={v['lift']:+.3f}  score={v['score']:.1f}")

ALL_VERIFIED = []
for label, seeds in SEED_LEXICONS.items():
    v = pool_for(seeds, CONTENT_HEADS)
    for r in v:
        r["group"] = label
    show(label, v)
    ALL_VERIFIED += [r for r in v if r["verified"]]

show("Self-copy baseline (control . head %d)" % CONTENT_HEADS[0],
     [{**v, "head": CONTENT_HEADS[0]} for v in st.verify_pool(
         model, st.candidate_pool(model, head=CONTENT_HEADS[0], freq=FREQ,
                                  include_self_copy=True, top_n=20))])

unsup = pool_for(None, CONTENT_HEADS)
for r in unsup:
    r["group"] = "Unsupervised"
show("Unsupervised - what else is in here", unsup)
ALL_VERIFIED += [r for r in unsup if r["verified"]]

# Distinct source->output vs raw verified count: the gap is dest-variant inflation (the §5c bug).
print("\nDistinct source->output per category (verified count inflates findings via dest-variants):")
for label in list(SEED_LEXICONS) + ["Unsupervised"]:
    vs = [v for v in ALL_VERIFIED if v.get("group") == label]
    nd = len({(v["source"], v["output"]) for v in vs})
    print(f"   {label}: {len(vs)} verified, {nd} distinct source->output")

# A triple is a GENUINE 3-way skip-trigram only if its DESTINATION is a real co-occurring token
# (not a spurious QK fragment). We curate those by hand — gloss keyed on the FULL (source, dest,
# output). A category with no glossable triple yields only dest-independent OV promotions (§5c).
GLOSS = {
    ("الرغم", "من", "أن"): "على الرغم من أن — concessive frame",
    ("تبلغ", "من", "العمر"): "تبلغ من العمر — 'is X years old'",
    ("رض", "ي", "الله"): "رضي الله — religious formula (BPE split رضي→رض+ي)",
    ("بالإضافة", "إلى", "ذلك"): "بالإضافة إلى ذلك — 'in addition'",
}
def glossed(v):
    return GLOSS.get((id_to_str.get(v["source"]), id_to_str.get(v["dest"]), id_to_str.get(v["output"])))

# Featured = the genuine frames. A category with no glossable triple is null *as a skip-trigram*
# (only the §5c dest-independent OV promotion fired), even if many triples technically verified.
featured, NULLS = [], []
for label in list(SEED_LEXICONS) + ["Unsupervised"]:
    vs = [v for v in ALL_VERIFIED if v.get("group") == label]
    g = [v for v in vs if glossed(v)]
    if g:
        featured.append(max(g, key=lambda v: v["lift"]))
    else:
        NULLS.append(label)
BOARD = sorted(ALL_VERIFIED, key=lambda v: v["lift"], reverse=True)[:12]
# Evidence for the split: the destinations each (source->output) actually verified with.
# A genuine frame's set contains a real co-occurring token (من, ي); a dest-independent OV
# promotion's set is only spurious QK fragments (حد/حض/فض/وض) — the §5c bug, made visible.
def _dests(source, output):
    ds = [id_to_str.get(v["dest"]) for v in ALL_VERIFIED
          if v["source"] == source and v["output"] == output]
    return list(dict.fromkeys(ds))  # order-preserving unique (a source may seed two categories)

print("\nDestination evidence (a genuine frame rides a real token; a bug rides a fragment bag):")
for v in featured:
    print(f"   FRAME ({id_to_str.get(v['source'])} -> {id_to_str.get(v['output'])}): "
          f"dests = {_dests(v['source'], v['output'])}")
for label in NULLS:
    seen, picks = set(), []
    for v in sorted((x for x in ALL_VERIFIED if x.get("group") == label),
                    key=lambda x: x["lift"], reverse=True):
        k = (v["source"], v["output"])
        if k not in seen:
            seen.add(k)
            picks.append(v)
        if len(picks) == 2:
            break
    for v in picks:
        print(f"   BUG   ({id_to_str.get(v['source'])} -> {id_to_str.get(v['output'])}): "
              f"dests = {_dests(v['source'], v['output'])}  [{label}]")

print(f"\ngenuine frames: {len(featured)}; no-frame categories (OV-promotion only): {NULLS or 'none'}")

<div dir="rtl">

### ٥ج. العيب البنيوي · The structural bug

الثلاثيات اللي اتأكّدت فوق **حقيقية** — بس كل واحد منها هو **أحسن شريحة** للرأس، مش اشتراط ثلاثي كامل. السبب: الـ `OV[source]` **مستقل عن الوجهة** — الوجهة بتقرر *هل* هتبص على الـ source، مش *إيه* اللي هيترفع. يبقى أي source قوي بيرفع **أكتر من ناتج** مهما كانت الوجهة. ده العيب الكلاسيكي بتاع الـ skip-trigram في *Elhage et al. (2021)*: `keep…in→mind` بيجرّ معاه `keep…in→bay`.

</div>

<div dir="ltr">

### 5c. The structural bug

The verified triples above are **real** — but each is the head's **best slice**, not full
three-way conditioning. Reason: `OV[source]` is **destination-independent** — the destination
decides *whether* you attend to the source, not *what* gets promoted. So a strong source
raises **several outputs** regardless of destination. This is the classic skip-trigram bug of
*Elhage et al. (2021)*: wanting `keep…in→mind` forces `keep…in→bay` along with it.

</div>

In [ ]:
# §5c — show the bug as a table: one source promotes a SECOND output regardless of destination.
# OV[source] is destination-independent, so wanting [.. -> output] drags [.. -> other] with it.
from IPython.display import HTML, display

ref = max((v for v in featured if glossed(v)), key=lambda v: v["lift"], default=None)
if ref is not None:
    _, OV = st.head_circuits(model, ref["head"])
    s = ref["source"]
    extra = [int(i) for i in torch.topk(OV[s, :FREQ], 4).indices
             if int(i) not in (s, ref["output"])]
    o_bug = extra[0] if extra else ref["output"]
    bug_rows = [
        {"source": s, "dest": ref["dest"], "output": ref["output"], "head": ref["head"],
         "group": "اللي عايزينه · wanted", "gloss": glossed(ref)},
        {"source": s, "dest": ref["dest"], "output": o_bug, "head": ref["head"],
         "group": "بيجي معاه · rides along", "gloss": "same OV[source] promotes it too"},
    ]
    display(HTML(st.triple_table_html(
        bug_rows, id_to_str,
        title="العيب البنيوي · the structural skip-trigram bug — one source, two outputs",
        note="الوجهة بتتحكم بس في *هل* الانتباه يوصل للمصدر، مش *إيه* اللي يترفع · the destination "
             "only gates WHETHER attention reaches the source, never WHICH output rises — so one "
             "source raises both (keep...in->mind forces keep...in->bay)")))
else:
    print("no glossable verified triple to illustrate the bug on (tiny/CI model).")

<div dir="rtl">

## ٦. خريطة الانتباه · Attention heatmap

دائرة الـ QK بتقرر نمط الانتباه. نشوفه لرأس واحد على جملة حقيقية: كل صف (وجهة) بيوزّع انتباهه على التوكنز اللي قبله. (بنعرض من اليمين للشمال زي العربي.)

</div>

In [ ]:
sent = "هو قال إنه هيسافر بكرة بدري عشان الشغل"
sids = encode(sent)[: model.cfg.n_ctx]
str_toks = [id_to_str.get(i, "[?]") for i in sids]
_, hcache = model.run_with_cache(torch.tensor([sids]).to(tiny.device()))
pattern = hcache["pattern", 0][0, 0].detach().cpu().numpy()  # head 0  <- (seq, seq)

fig = go.Figure(go.Heatmap(z=pattern, x=str_toks, y=str_toks, colorscale="Blues"))
fig.update_layout(title="Head 0 attention — " + sent, xaxis=dict(side="top"), height=460, width=560)
fig.update_xaxes(autorange="reversed")  # RTL
fig.update_yaxes(autorange="reversed")
fig.show()

<div dir="rtl">

## ٧. معبّر بشكل مفاجئ · Surprisingly expressive

دي الخلاصة (punchline) — بس مع تحذير أمين: العدّ الخام للثلاثيات المتأكّدة **بيضخّم** عدد النتايج، لأن كل ترقية `source→output` بتتكرّر مع كذا وجهة، وأغلبها شظايا QK مالهاش معنى (§٥ج). فبنعرض بس **الأطر الحقيقية**: اللي وجهتها توكن حقيقي بيجي فعلًا قبل الناتج في النص. كل سطر فيه تفسير لغوي — ده اللي يخلّيه «معبّر». الجدول التاني = أعلى ١٢ بالرفع (خام، من غير فلترة)، والرقم المجمّع اتنقل لحاشية.

</div>

<div dir="ltr">

## 7. Surprisingly expressive

This is the punchline — with an honest caveat. The raw count of verified triples **inflates** the
findings, because each `source→output` OV-promotion repeats across several destinations, most of
them meaningless QK fragments (§5c). So we feature only the **genuine frames**: triples whose
destination is a real token that actually precedes the output in text. Each carries a gloss —
that's what makes it *expressive*. The second table is the raw top-12 by lift (unfiltered); the
aggregate "how much does context move things" number is demoted to a footnote.

</div>

In [ ]:
# §7 — the punchline as paper-style tables: the genuine 3-way frames (meaningful destination),
# then the raw top-12 board. The aggregate context-shift is demoted to a footnote.
from IPython.display import HTML, display

def with_gloss(v):
    return {**v, "gloss": glossed(v) or ""}

# aggregate (footnote, not headline): how far does the skip-trigram move the bigram distribution?
W = min(16, model.cfg.n_ctx)
flat = encode(EVAL_TEXT)
wins = [flat[i:i + W] for i in range(0, len(flat) - W, 4)][:32]
batch = torch.tensor(wins).to(tiny.device())
with torch.no_grad():
    logits2, cache2 = model.run_with_cache(batch)
    direct2 = cache2["resid_pre", 0] @ model.W_U + model.b_U
    p_full, p_dir = torch.softmax(logits2, -1), torch.softmax(direct2, -1)
reshape = float((p_full - p_dir).abs().max(-1).values.mean())
changed = float((p_full.argmax(-1) != p_dir.argmax(-1)).float().mean())

null_note = ("بلا إطار حقيقي — ترقيات OV مستقلة عن الوجهة بس (§٥ج): " + "، ".join(NULLS)) if NULLS \
    else "كل التصنيفات طلّعت إطار حقيقي · every category produced a genuine frame"
display(HTML(st.triple_table_html(
    [with_gloss(v) for v in featured], id_to_str,
    title="أطر حقيقية (وجهة ذات معنى) · genuine 3-way frames (meaningful destination)",
    note="من ~١٠٠ مرشّح/تصنيف، بعد إسقاط الترقيات المستقلة عن الوجهة · genuine frames only, after "
         "dropping dest-independent promotions.  " + null_note)))
display(HTML(st.triple_table_html(
    [with_gloss(v) for v in BOARD], id_to_str,
    title="أعلى ١٢ بالرفع (خام) · raw top-12 by lift (unfiltered)",
    note=f"أعلى رفع غالبًا الموديل بيكمّل كلمة BPE متكسّرة (الاكت→ابات)، وكتير من الباقي وجهته شظية · "
         f"top lifts are often the model completing a BPE-fragmented word, and many others have a "
         f"fragment destination.  السياق بيحرّك التوزيع بمتوسط {reshape:.3f} ويقلب التوكن رقم ١ في "
         f"{changed * 100:.0f}% · context reshapes the dist by avg {reshape:.3f}, flips top-1 at "
         f"{changed * 100:.0f}% of positions")))

<div dir="rtl">

## ٨. هل القصة بتصمد مع توكنايزر تاني؟ · BPE مقابل Unigram

نفس المعمار، نفس الكوربس، **توكنايزر مختلف**. الفرضية الأمينة: الـ skip-trigrams **نتاج التوكنة** — حبيبة التوكنايزر هي اللي بتحدّد أنهي تراكيب أصلًا ممكن تتعلّم كـ skip-trigram. بنحمّل التوأم المدرّب بالـ Unigram (نفس البيانات) ونعيد **نفس التأكيد المحتجوز** بالظبط. لو القصة بتاعت §٧ صرف خصائص الموديل، المفروض تتكرّر؛ لو هي نتاج التوكنايزر، المفروض **تتغيّر**.

</div>

<div dir="ltr">

## 8. Does the story survive a different tokenizer? — BPE vs Unigram

Same architecture, same corpus, **different tokenizer**. The honest hypothesis: skip-trigrams are
a **function of tokenization** — the tokenizer's granularity decides which constructions can be
learned as a skip-trigram *at all*. We load the unigram-tokenised twin (same data) and re-run the
**identical held-out verification**. If §7's story were purely about the model, it should
reproduce; if it's a tokenizer effect, it should **change**.

</div>

In [ ]:
# §8 — re-run the SAME held-out verification on the unigram-tokenised twin checkpoint.
# Loads local checkpoints/stage2dash_unigram/ if present, else pulls the twin from the Hub
# (HF_REPO_UNIGRAM) so this works on Colab too. Auto-skips under FORCE_TINY (tiny CI model) or
# if the twin can't be fetched, so the notebook still completes. Nothing global is clobbered.
from IPython.display import HTML, display

UNI_DIR = os.path.join(os.path.dirname(CKPT_DIR), "stage2dash_unigram")
HF_REPO_UNIGRAM = "yassermakram/fanous-stage2dash-attn-only-1l-unigram"

umodel = None
if globals().get("FORCE_TINY"):
    print("§8 skipped under FORCE_TINY (the tiny CI model has no unigram twin).")
else:
    try:
        from tokenizers import Tokenizer
        utpath = os.path.join(UNI_DIR, "tokenizer.json")
        umpath = os.path.join(UNI_DIR, "model.pt")
        if not (os.path.exists(utpath) and os.path.exists(umpath)):
            from huggingface_hub import hf_hub_download
            utpath = hf_hub_download(HF_REPO_UNIGRAM, "tokenizer.json")
            umpath = hf_hub_download(HF_REPO_UNIGRAM, "model.pt")
        uckpt = torch.load(umpath, map_location=tiny.device(), weights_only=False)
        umodel = _model_from_ckpt(uckpt)
        utok = Tokenizer.from_file(utpath)
    except Exception as e:
        print(f"§8 skipped: could not load the unigram twin (local {UNI_DIR} or "
              f"HF {HF_REPO_UNIGRAM}) — {type(e).__name__}: {e}")

if umodel is not None:
    uVOCAB = utok.get_vocab_size()
    u_id_to_str = {i: (utok.id_to_token(i) or "[?]") for i in range(uVOCAB)}
    def uencode(t):
        return utok.encode(t).ids
    uFREQ = min(2500, uVOCAB)

    # (a) THE MECHANISM, shown: how each tokenizer segments the seeds that mattered in §5-§7.
    seeds_to_show = ["على", "الرغم", "رضي", "تبلغ", "ال", "الذي", "اللي", "عايز", "دلوقتي"]
    rows = "".join(
        f"<tr><td dir='rtl'><b>{w}</b></td>"
        f"<td dir='ltr'>{' '.join(id_to_str.get(i, '[?]') for i in encode(w))}</td>"
        f"<td dir='ltr'>{' '.join(u_id_to_str.get(i, '[?]') for i in uencode(w))}</td></tr>"
        for w in seeds_to_show)
    display(HTML("<table dir='rtl' border='1' cellpadding='5' style='border-collapse:collapse'>"
                 "<tr><th>seed</th><th>BPE</th><th>Unigram</th></tr>" + rows + "</table>"))

    # (b) the SAME pipeline on the unigram model: content heads -> pool -> held-out verify.
    _, uhc = umodel.run_with_cache(
        torch.tensor([uencode(EVAL_TEXT)[:umodel.cfg.n_ctx]]).to(tiny.device()))
    u_heads = [h for h in range(umodel.cfg.n_heads)
               if st.head_attention_kind(uhc["pattern", 0][0, h].detach().cpu()) == "content"] or [0]

    def upool(seed_words):
        src = set(st.seed_ids(uencode, u_id_to_str, seed_words, uFREQ)) if seed_words else None
        po, pd = (5, 4) if seed_words else (1, 1)
        cands = []
        for h in u_heads:
            for v in st.candidate_pool(umodel, head=h, freq=uFREQ, top_n=100, sources=src,
                                       per_source_outputs=po, per_source_dests=pd):
                cands.append({**v, "head": h})
        cands = st.dedup_triples(cands)
        cands.sort(key=lambda c: c["score"], reverse=True)
        return [st.verify_triple(umodel, c) for c in cands[:100]]

    ALL_U = []
    for label, seeds in list(SEED_LEXICONS.items()) + [("Unsupervised", None)]:
        for r in upool(seeds):
            if r["verified"]:
                r["group"] = label
                ALL_U.append(r)
    U_BOARD = sorted(ALL_U, key=lambda v: v["lift"], reverse=True)[:12]

    # (c) one objective contrast: share of verified triples with a SINGLE-CHARACTER destination.
    # Unigram fragments Arabic to characters, so its skip-trigrams are far more sub-lexical.
    def char_dest_share(rows_, idmap):
        return sum(len(idmap.get(r["dest"], "")) == 1 for r in rows_) / max(1, len(rows_))
    print(f"content heads (unigram): {u_heads}")
    print(f"single-character destination share — "
          f"BPE {char_dest_share(ALL_VERIFIED, id_to_str):.0%}  "
          f"vs  Unigram {char_dest_share(ALL_U, u_id_to_str):.0%}")

    display(HTML(st.triple_table_html(
        [{**v, "gloss": ""} for v in U_BOARD], u_id_to_str,
        title="Unigram — أعلى ١٢ بالرفع (خام) · raw top-12 by lift",
        note="نفس التأكيد المحتجوز، توكنايزر Unigram · same held-out verification, unigram "
             "tokenizer. سلسلة الجمع المذكّر السالم ـون (…ون → الاسم الجمع التالي) وإكمال الحروف هي "
             "المهيمنة هنا — مش الأطر المعجمية اللي طلّعها BPE · the ـون masculine-plural concord "
             "chain and character completion dominate here, not the lexical MSA frames BPE found.")))

<div dir="rtl">

**القراءة الأمينة:** التوكنايزر بيختار القواعد اللي الـ skip-trigram يقدر يعبّر عنها.

- الـ BPE بيقسّم «رضي»→«رض»+«ي»، فبيستضيف الإطار المعجمي **رضي الله**؛ الـ Unigram بيخلّي «رضي» كلمة واحدة، فالإطار ده **مايقدرش يوجد** عنده أصلًا.
- الـ Unigram بيكسّر العربي لحروف (ال→ا+ل، عايز→٤ توكنز)، فأطره **تحت-معجمية**: سلسلة الجمع المذكّر السالم ـون (عسكريو…ن→أمريكيون، ملاكمو…ن→أولمبيون) وإكمال الحروف (الرغم…م→ن = «من»). نسبة الوجهة أحادية-الحرف فوق بتأكّد ده برقم.
- **الخلاصة:** نفس الموديل ونفس الكوربس، توكنايزر مختلف ⇒ **skip-trigrams مختلفة**. الـ BPE → أطر معجمية/اصطلاحية؛ الـ Unigram → أطر صرفية/حرفية. القصة بتاعت §٧ مش خاصية موديل مجرّدة؛ هي **مشروطة بالتوكنة**.

</div>

<div dir="ltr">

**Honest reading:** the tokenizer chooses which grammar a skip-trigram can express.

- BPE splits رضي→رض+ي, so it hosts the lexical frame **رضي الله**; Unigram keeps رضي whole, so
  that frame **cannot exist** there at all.
- Unigram fragments Arabic to characters (ال→ا+ل, عايز→4 tokens), so its frames are
  **sub-lexical**: the masculine-plural ـون concord chain (عسكريو…ن→أمريكيون, ملاكمو…ن→أولمبيون)
  and character completion (الرغم…م→ن = "من"). The single-character-destination share above
  quantifies this.
- **Bottom line:** same model, same corpus, different tokenizer ⇒ **different skip-trigrams**.
  BPE → lexical/idiomatic frames; Unigram → morphological/character frames. §7's story is not an
  abstract property of the model — it is **conditioned on tokenization**.

</div>

<div dir="rtl">

## ٩. الخلاصة · Recap (اللي طلع فعلًا)

- **التفكيك مظبوط:** الـ bigram + skip-trigram = مخرج الموديل بالظبط (الفرق ~١e‑٥).
- **التشخيص:** على جملة التشخيص الواحدة، كل الـ ٨ رؤوس طلعت **content**. مربوط بالجملة دي.
- **التأكيد بيشتغل — بس العدّ بيخدع:** كل تصنيف مبذور طلّع عشرات الثلاثيات المتأكّدة (٤٨/٥٢/٤٩/٤٩)، لكن دي **متضخّمة**: `per_source_dests=4` بيجوّز كل ترقية `source→output` مع لحد ٤ وجهات، وبما إن `OV[source]` مستقل عن الوجهة (§٥ج) نفس الترقية بتتأكّد تحت كذا وجهة. المتمايز فعلًا (source→output) أقل: ٢٩/٢٣/١٧/١٦. الغير موجَّه (١×١) مش متضخّم: ٧٥=٧٥.
- **الأطر الحقيقية (وجهتها توكن له معنى) — دي النتيجة:**
  - **الرغم … من → أن** (+٠.٢٧، رأس ٦) = «على الرغم من أن».
  - **رض … ي → الله** (+٠.٥٥، رأس ٢) = «رضي الله» (الـ BPE قسّم رضي→رض+ي).
  - **تبلغ … من → العمر** (+٠.٩٨، رأس ٦) = «تبلغ من العمر».
  - (+ **بالإضافة … إلى → ذلك** +٠.٠٧).
- **أداة التعريف والتباين = مفيش إطار حقيقي:** اللي اتأكّد هناك كله ترقية OV مستقلة عن الوجهة (التي→ها/فيها لأي شظية وجهة). ده **صرف حقيقي في دائرة الـ OV** بس مش skip-trigram ثلاثي — يعني التصنيفين دول **فاضيين كـ skip-trigram** (بصحّح ادعائي السابق إنهم «مش فاضيين»). والمصري «اللي» غايب خالص؛ التباين طلّع الفصحى «الذي» بس — فالإشارة اللهجية ضعيفة، متسق مع كوربَس فصحى-أغلب.
- **أعلى رفع = إكمال كلمة متكسّرة:** أقوى الثلاثيات الخام (الاكت→ابات +٠.٩٨، الإ→فاقية +٠.٨٩) هي الموديل بيكمّل كلمة قسّمها الـ BPE — نتيجة عن التوكنايزر.
- **اختيار الرأس بيفرق:** الأطر عايشة على رؤوس ٢/٦ (وترقيات الصرف على ١)؛ الرأس ٠ **مهيمن عليه النسخ** (الضابط: هوكي→هوكي +٠.٩٣، سانت→سانت +٠.٨٩).
- **العيب البنيوي هو القصة، مش حاشية:** المصدر القوي بيرفع أكتر من ناتج بصرف النظر عن الوجهة (تبلغ → العمر + أمريكية) — وده بالظبط سبب إن أغلب «المتأكّد» مش إطار حقيقي.
- **مجمّع:** الـ skip-trigram بيعيد تشكيل التوزيع بمتوسط ٠.١٢ وبيقلب التوكن رقم ١ في ٨٩% من المواضع.

- **التوكنايزر بيحدّد القاعدة (§٨):** التوأم بالـ Unigram **مافيهوش** إطار معجمي زي
  رضي الله — الـ Unigram بيخلّي رضي كلمة واحدة — وبيطلّع بدلها سلسلة الجمع المذكّر
  السالم ـون (عسكريو…ن→أمريكيون) وإكمال الحروف؛ ونسبة الوجهة أحادية-الحرف بتقفز من
  BPE لـ Unigram. **الـ skip-trigrams نتاج التوكنة**، مش خاصية مجرّدة للموديل.

الخلاصة الأمينة: الموديل بيستضيف **عدد صغير من الأطر الحقيقية المقروءة** للفصحى (على الرغم من أن، رضي الله، يبلغ من العمر) على رؤوس معيّنة — وأغلب «المتأكّد» التاني هو ترقية OV مستقلة عن الوجهة (العيب البنيوي) أو إكمال كلمة متكسّرة، والإشارة اللهجية لسه ضعيفة.

</div>

<div dir="ltr">

## 9. Recap (what actually came out)

- **Decomposition is exact:** bigram + skip-trigram reproduces the model logits (diff ≈ 1e‑5).
- **Diagnosis:** on the single diagnosis sentence all 8 heads classify as **content** (tied to
  that sentence).
- **Verification works — but the count deceives:** each seeded category yields dozens of verified
  triples (48/52/49/49), but these are **inflated**: `per_source_dests=4` crosses each
  `source→output` OV-promotion with up to 4 destinations, and since `OV[source]` is
  destination-independent (§5c) the same promotion verifies under several dests. Distinct
  `source→output` is lower: **29/23/17/16**. The unsupervised pool (1×1) is not inflated: 75 = 75.
- **The genuine frames (destination is a meaningful token) — this is the result:**
  - **الرغم … من → أن** (+0.27, head 6) = *على الرغم من أن* ("despite that").
  - **رض … ي → الله** (+0.55, head 2) = *رضي الله* (BPE split رضي→رض+ي).
  - **تبلغ … من → العمر** (+0.98, head 6) = "is X years old".
  - (+ **بالإضافة … إلى → ذلك** +0.07).
- **Definite-article & contrast = no genuine frame:** everything that verified there is a
  destination-independent OV promotion (التي→ها/فيها for any fragment dest). That is **real
  morphology in the OV circuit** but *not* a three-way skip-trigram — so both categories are
  **null as skip-trigrams** (correcting my earlier overclaim that they were "not null"). Masri
  اللي is absent entirely; the contrast category surfaces only MSA الذي — the dialect signal
  stays weak, consistent with an MSA-heavy corpus.
- **Top lift = completing a fragmented word:** the strongest raw triples (الاكت→ابات +0.98,
  الإ→فاقية +0.89) are the model completing an MSA word that BPE split — a tokenizer effect.
- **Head choice matters:** the frames live on heads 2/6 (morphology promotions on head 1); head 0
  is **copy-dominated** (control: هوكي→هوكي +0.93, سانت→سانت +0.89).
- **The structural bug is the story, not a footnote:** a strong source raises several outputs
  regardless of destination (تبلغ → العمر *and* أمريكية) — which is exactly why most "verified"
  triples are not genuine frames.
- **Aggregate:** the skip-trigram reshapes the next-token distribution by avg 0.12 and flips the
  top-1 prediction at 89% of positions.

- **Tokenizer decides the grammar (§8):** the unigram twin hosts *no* lexical
  frame like رضي الله — unigram keeps رضي whole — and instead surfaces the
  masculine-plural ـون concord chain (عسكريو…ن→أمريكيون) and character completion;
  the single-character-destination share jumps from BPE to Unigram. **Skip-trigrams
  are a function of tokenization**, not an abstract property of the model.

Honest bottom line: the model hosts a **small number of legible, genuine MSA frames** (على الرغم
من أن, رضي الله, يبلغ من العمر) on specific heads — while most other "verified" triples are
destination-independent OV promotions (the structural bug) or completions of BPE-fragmented
words, and the dialect signal stays weak.

</div>